# 03 - Review duplicates and lock leakage-aware splits

Run on Colab after notebook 02. CPU only. Reads the audit manifest from Drive; no image download, training or quality filtering.

Policy: FVC2000/2002 DB1-DB3 are development data; FVC2004 DB1-DB3 remain external test. All impressions of a scoped finger stay together. Scoped IDs connected by exact duplicate images share a leakage group (not an assertion that they are the same finger). Source groups overlapping external test are quarantined in their entirety. No image is deleted and all original FVC2004 images remain in external test.

Development groups are assigned deterministically with seed 20261110 and nominal probabilities 60/20/20. Counts are approximate, not guaranteed exact quotas. Official A/B labels are retained as metadata; these custom splits are NOT the official FVC protocol.

Limitations: byte hashes cannot detect transformed copies or establish subject independence across databases/years. No subject IDs are available in this manifest. Dry/wet labels remain unknown. Duplicate images inside a test set may bias metrics: inspect the duplicate report and later report a duplicate-aware sensitivity analysis as well as unmodified benchmark results.

Outputs are frozen in MyDrive/FingerprintResearch/protocols/fvc_split_v1. Reruns must produce exactly the same files or stop; changing a frozen protocol requires an explicitly new version.

In [ ]:
import hashlib
import json
from pathlib import Path

import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
AUDIT_DIR = PROJECT_ROOT / 'reports' / 'fvc_audit_v1'
PROTOCOL_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
MANIFEST_PATH = AUDIT_DIR / 'manifest_real_images.csv'
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError('Run notebook 02 first; audit manifest is missing.')
audit_bytes = MANIFEST_PATH.read_bytes()
audit_sha256 = hashlib.sha256(audit_bytes).hexdigest()
manifest = pd.read_csv(MANIFEST_PATH, keep_default_na=False)
required = {'dataset', 'db', 'official_subset', 'finger_id', 'impression_id',
            'identity_key', 'relative_path', 'condition', 'split', 'sha256'}
if not required.issubset(manifest.columns):
    raise RuntimeError(f'Missing columns: {required - set(manifest.columns)}')
if len(manifest) != 7920 or manifest['relative_path'].duplicated().any():
    raise RuntimeError('Unexpected image count or duplicate paths.')
expected_datasets = {'FVC2000', 'FVC2002', 'FVC2004'}
if set(manifest['dataset']) != expected_datasets or set(manifest['db']) != {'DB1', 'DB2', 'DB3'}:
    raise RuntimeError('Unexpected datasets or sensor databases.')
expected_ids = (manifest['dataset'] + '_' + manifest['db'] + '_'
                + manifest['finger_id'].astype(int).astype(str).str.zfill(3))
if not manifest['identity_key'].eq(expected_ids).all():
    raise RuntimeError('Identity keys do not match dataset/database/finger IDs.')
if manifest.duplicated(['identity_key', 'impression_id']).any():
    raise RuntimeError('Duplicate identity/impression keys.')
sizes = manifest.groupby('identity_key').size()
if len(sizes) != 990 or not sizes.eq(8).all():
    raise RuntimeError('Expected 990 scoped finger IDs with 8 impressions each.')
counts = manifest.groupby(['dataset', 'db', 'official_subset']).size().to_dict()
expected_counts = {(year, db, subset): 800 if subset == 'A' else 80
                   for year in sorted(expected_datasets)
                   for db in ('DB1', 'DB2', 'DB3') for subset in ('A', 'B')}
if counts != expected_counts:
    raise RuntimeError('Audit subset counts do not match the official archive.')
if not manifest['sha256'].str.fullmatch(r'[0-9a-f]{64}').all():
    raise RuntimeError('Invalid image hashes.')
if not manifest['condition'].eq('unknown').all() or not manifest['split'].eq('unassigned').all():
    raise RuntimeError('Expected unlabelled, unassigned audit input from notebook 02.')
duplicates = manifest[manifest.duplicated('sha256', keep=False)].sort_values(
    ['sha256', 'identity_key', 'impression_id'])
print('Audit manifest SHA256:', audit_sha256)
print('Exact duplicate image rows:', len(duplicates))
print(duplicates[['dataset', 'db', 'identity_key', 'impression_id', 'relative_path', 'sha256']].to_string(index=False))

In [ ]:
SEED = 20261110

def assign_splits(frame, seed):
    result = frame.copy()
    identities = sorted(result['identity_key'].unique())
    parent = {identity: identity for identity in identities}

    def root(identity):
        while parent[identity] != identity:
            parent[identity] = parent[parent[identity]]
            identity = parent[identity]
        return identity

    for _, group in result.groupby('sha256', sort=True):
        members = sorted(group['identity_key'].unique())
        for member in members[1:]:
            left, right = sorted((root(members[0]), root(member)))
            parent[right] = left
    result['leakage_group'] = result['identity_key'].map(root)
    external = result['dataset'].eq('FVC2004')
    external_groups = set(result.loc[external, 'leakage_group'])

    def development_split(group):
        value = int.from_bytes(hashlib.sha256(f'{seed}:{group}'.encode('ascii')).digest()[:8], 'big')
        if value < int(0.6 * 2**64):
            return 'train'
        if value < int(0.8 * 2**64):
            return 'validation'
        return 'test_synthetic'

    result['split'] = result['leakage_group'].map(development_split)
    result.loc[result['leakage_group'].isin(external_groups), 'split'] = 'quarantine_external_overlap'
    result.loc[external, 'split'] = 'test_external'
    return result.sort_values(['dataset', 'db', 'finger_id', 'impression_id']).reset_index(drop=True)

locked = assign_splits(manifest, SEED)
active = locked[~locked['split'].eq('quarantine_external_overlap')]
if active.groupby('sha256')['split'].nunique().gt(1).any():
    raise RuntimeError('Exact image copies cross active splits.')
if locked.groupby('identity_key')['split'].nunique().gt(1).any():
    raise RuntimeError('Impressions of the same scoped finger cross splits.')
if not locked.loc[locked['dataset'].eq('FVC2004'), 'split'].eq('test_external').all():
    raise RuntimeError('FVC2004 was modified or assigned to development.')
if locked['split'].eq('test_external').sum() != 2640:
    raise RuntimeError('External benchmark count changed.')
for key, group in locked[~locked['dataset'].eq('FVC2004')].groupby(['dataset', 'db']):
    if not {'train', 'validation', 'test_synthetic'}.issubset(set(group['split'])):
        raise RuntimeError(f'A development split is empty in {key}. Review before freezing.')

summary = locked.groupby(['dataset', 'db', 'split']).agg(
    images=('relative_path', 'size'), scoped_fingers=('identity_key', 'nunique'),
    leakage_groups=('leakage_group', 'nunique')
).reset_index()
print(summary.to_string(index=False))
print('\nQuarantined images:', locked['split'].eq('quarantine_external_overlap').sum())
print('PASS: no scoped finger or byte-identical image crosses active splits.')

In [ ]:
def csv_bytes(frame):
    return frame.to_csv(index=False, lineterminator='\n').encode('utf-8')

duplicate_report = locked[locked.duplicated('sha256', keep=False)].sort_values(
    ['sha256', 'identity_key', 'impression_id'])
files = {
    'manifest_splits.csv': csv_bytes(locked),
    'split_counts.csv': csv_bytes(summary),
    'duplicate_review.csv': csv_bytes(duplicate_report),
}
protocol = {
    'version': 'fvc_split_v1', 'seed': SEED,
    'audit_manifest_sha256': audit_sha256,
    'development_datasets': ['FVC2000', 'FVC2002'],
    'external_dataset': 'FVC2004', 'dbs': ['DB1', 'DB2', 'DB3'],
    'nominal_development_probabilities': {'train': 0.6, 'validation': 0.2, 'test_synthetic': 0.2},
    'assignment': 'first 64 SHA256 bits of seed:leakage_group, thresholds 0.6 and 0.8',
    'duplicate_policy': 'connect scoped IDs; quarantine source groups overlapping external test',
    'identity_scope': 'dataset + database + finger; not verified subject-independent',
    'condition_labels': 'unknown; do not infer dry/wet',
    'official_fvc_protocol': False,
    'file_sha256': {name: hashlib.sha256(data).hexdigest() for name, data in files.items()},
}
files['protocol.json'] = (json.dumps(protocol, indent=2, sort_keys=True) + '\n').encode('utf-8')

def freeze_files(directory, contents):
    directory.mkdir(parents=True, exist_ok=True)
    for name, data in contents.items():
        destination = directory / name
        if destination.exists() and destination.read_bytes() != data:
            raise RuntimeError(f'Frozen protocol differs: {destination}. Do not overwrite or delete it.')
    for name, data in contents.items():
        destination = directory / name
        if not destination.exists():
            with destination.open('xb') as output:
                output.write(data)
        if destination.read_bytes() != data:
            raise RuntimeError(f'Persistence verification failed: {destination}')

freeze_files(PROTOCOL_DIR, files)
print('Protocol frozen and read-back verified:', PROTOCOL_DIR)
print('Next: establish an original-image matcher baseline. No model training yet.')
print('Save this notebook in VS Code; protocol outputs are on Drive.')